# A faster `np.unique`

`main` against a new hash set and lookup tables.

In [ ]:
import json
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

HERE = Path.cwd()
BUILDS = {name: HERE / f".pixi/envs/{name}/bin/python" for name in ("main", "tables")}
COLORS = {"main": "#9aa0a6", "tables": "#1a73e8", "six lines of Python\non main": "#188038"}
DATA = HERE / "data"
DATA.mkdir(exist_ok=True)
plt.rcParams.update({"font.size": 15, "axes.spines.top": False, "axes.spines.right": False})

# Runs in each build: time a call a few times, and record how far the
# process's memory rises above its level before the first call.
WORKER = r"""
import json, resource, sys, time
import numpy as np

def peak_rss():
    r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return r if sys.platform == "darwin" else r * 1024

a = np.load(sys.argv[1])
exec(sys.argv[4])
call = eval("lambda a: " + sys.argv[2])
before = peak_rss()
times = []
for _ in range(int(sys.argv[3])):
    t0 = time.perf_counter()
    out = call(a)
    out = out[0] if isinstance(out, tuple) else out
    times.append(time.perf_counter() - t0)
    if len(times) == 1:
        extra = peak_rss() - before
print(json.dumps({"seconds": min(times), "extra_bytes": extra, "distinct": int(out.size),
                  "numpy": np.__version__}))
"""


def bench(array, call="np.unique(a)", repeats=3, define="", builds=None):
    """Time `call` in each build, on an array or on the path of a .npy file."""
    path = array if isinstance(array, Path) else DATA / "tmp_array.npy"
    if path is not array:
        np.save(path, array)
    results = {}
    for name in builds or BUILDS:
        python = BUILDS[name]
        out = subprocess.run([python, "-c", WORKER, path, call, str(repeats), define],
                             capture_output=True, text=True, check=True, cwd=DATA)
        results[name] = json.loads(out.stdout)
    if path is not array:
        path.unlink()
    return results


def speedups(results):
    base = results["main"]["seconds"]
    return ", ".join(f"{name} {base / r['seconds']:.1f}x" for name, r in results.items() if name != "main")


def show_labels(image, title):
    """Draw a slice of a label volume, one colour per label, background white."""
    _, codes = np.unique(image, return_inverse=True)
    codes = np.ma.masked_where(image == 0, codes.reshape(image.shape) * 7919 % 20)
    plt.figure(figsize=(6, 6))
    plt.imshow(codes, cmap="tab20", interpolation="nearest")
    plt.axis("off")
    plt.title(title, loc="left")
    plt.show()


def bars(values, unit, title):
    fig, ax = plt.subplots(figsize=(8, 2.8))
    names = list(values)
    ax.barh(names, [values[n] for n in names], color=[COLORS[n] for n in names])
    fig.set_figheight(0.75 * len(names) + 0.9)
    ax.invert_yaxis()
    ax.set_xlabel(unit)
    ax.set_title(title, loc="left")
    for i, n in enumerate(names):
        shown = f"{values[n]:,.0f}" if values[n] >= 100 else f"{values[n]:.3g}"
        ax.text(values[n], i, f"  {shown} {unit}", va="center")
    ax.set_xlim(0, max(values.values()) * 1.25)
    plt.show()


for name, python in BUILDS.items():
    version = subprocess.run([python, "-c", "import numpy; print(numpy.__version__)"],
                             capture_output=True, text=True, check=True).stdout.strip()
    print(f"{name:6s} numpy {version}   .pixi/envs/{name}")

## How fast does an embryo grow?

*C. elegans*: 385 3D frames, every nucleus labelled.

In [ ]:
# Run this cell before the talk (about 30 seconds). It reads both movies into shared memory,
# 8.6 GB, where each build finds them already loaded, and prepares the movie player.
import base64
import io
import time
from multiprocessing import shared_memory

import tifffile
from IPython.display import HTML, display
from matplotlib import colormaps
from PIL import Image

FRAME_STEP = 1      # 2 loads every other frame: half the memory, half the waiting below

for block in globals().get("SHARED", {}).values():      # free the memory of an earlier run
    block.close()
    block.unlink()
SHARED, MOVIES = {}, {}
for name, folder in (("embryo 1", "01"), ("embryo 2", "02")):
    files = sorted((DATA / "celegans" / folder).glob("man_seg*.tif"))[::FRAME_STEP]
    first = tifffile.imread(files[0])
    shape = (len(files), *first.shape)
    SHARED[name] = shared_memory.SharedMemory(create=True, size=int(np.prod(shape)) * first.itemsize)
    MOVIES[name] = np.ndarray(shape, dtype=first.dtype, buffer=SHARED[name].buf)
    for i, file in enumerate(files):
        MOVIES[name][i] = tifffile.imread(file)
    print(f"{name}: {shape[0]} frames of {shape[1:]} {first.dtype}, {MOVIES[name].nbytes / 2**30:.1f} GB")

# the player: the microscope image beside the labels, one JPEG per frame
raw = np.load(HERE / "assets/embryo1_projections.npz")["projections"][::FRAME_STEP]
magma = (colormaps["magma"](np.linspace(0, 1, 256))[:, :3] * 255).astype(np.uint8)
label_colours = np.vstack([[0, 0, 0], (colormaps["tab20"](np.arange(1, 4000) * 7 % 20)[:, :3] * 255)]).astype(np.uint8)
jpegs = []
for image, labels in zip(raw, MOVIES["embryo 1"]):
    side_by_side = np.hstack([magma[image], label_colours[labels.max(axis=0)[::2, ::2]]])
    buffer = io.BytesIO()
    Image.fromarray(side_by_side).save(buffer, format="JPEG", quality=85)
    jpegs.append(base64.b64encode(buffer.getvalue()).decode())
PLAYER = """
<div style="font: 16px sans-serif; max-width: 900px">
  <img id="embryo-frame" style="width: 100%; display: block">
  <div style="display: flex; align-items: center; gap: 12px; margin-top: 8px">
    <button id="embryo-play" style="font-size: 16px; padding: 4px 16px">&#9654; play</button>
    <input id="embryo-slider" type="range" min="0" max="LAST_FRAME" value="0" style="flex: 1">
    <span id="embryo-label" style="min-width: 6em">frame 0</span>
  </div>
</div>
<script>
(function () {
  const frames = FRAME_DATA, step = FRAME_STEP;
  const image = document.getElementById("embryo-frame"), slider = document.getElementById("embryo-slider");
  const label = document.getElementById("embryo-label"), button = document.getElementById("embryo-play");
  let timer = null;
  function show(i) {
    image.src = "data:image/jpeg;base64," + frames[i];
    slider.value = i;
    label.textContent = "frame " + i * step;
  }
  function stop() { clearInterval(timer); timer = null; button.innerHTML = "&#9654; play"; }
  button.onclick = function () {
    if (timer) { stop(); return; }
    if (+slider.value >= frames.length - 1) { show(0); }
    button.innerHTML = "&#10074;&#10074; pause";
    timer = setInterval(function () {
      if (+slider.value >= frames.length - 1) { stop(); } else { show(+slider.value + 1); }
    }, 60);
  };
  slider.oninput = function () { show(+slider.value); };
  show(0);
})();
</script>
""".replace("LAST_FRAME", str(len(jpegs) - 1)).replace("FRAME_DATA", json.dumps(jpegs)).replace("FRAME_STEP", str(FRAME_STEP))
print(f"movie: {len(jpegs)} frames, {len(PLAYER) / 2**20:.1f} MB")

In [ ]:
import os

# The player needs a live notebook. With DEMO_STATIC=1 in the environment four frames are shown
# instead, for the executed copy in the repository: GitHub does not run the player's script.
if os.environ.get("DEMO_STATIC"):
    fig, axes = plt.subplots(2, 2, figsize=(14, 5.6))
    for ax, i in zip(axes.flat, np.linspace(0, len(jpegs) - 1, 4).astype(int)):
        ax.imshow(Image.open(io.BytesIO(base64.b64decode(jpegs[i]))))
        ax.set_title(f"frame {i * FRAME_STEP}", loc="left")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    display(HTML(PLAYER))

One `np.unique` per frame, with the volume of every nucleus.

In [ ]:
# Runs in one build: count the labels of every frame, timing only np.unique, and report each frame as it is done.
MOVIE_WORKER = r"""
import json, os, sys, time
from multiprocessing import shared_memory
import numpy as np
movies = []
for name, dtype, shape in json.loads(sys.argv[1]):
    block = shared_memory.SharedMemory(name=name, track=False)
    movies.append((block, np.ndarray(shape, dtype=dtype, buffer=block.buf)))
print("ready", flush=True)
seconds = 0.0
for index, (block, movie) in enumerate(movies):
    for frame in movie:
        frame.reshape(-1)[::2048].sum()                    # map the frame's pages into this process
        t0 = time.perf_counter()
        ids, volumes = np.unique(frame, return_counts=True)  # each nucleus and its volume in voxels
        cells = ids.size - 1                               # label 0 is the background
        seconds += time.perf_counter() - t0
        print(index, cells, seconds, flush=True)
os._exit(0)
"""
EMBRYO_COLOURS = ["#1a73e8", "#e8710a"]


def draw(build, counts, seconds, done):
    """The growth curves so far, a progress bar and the time spent in np.unique, as SVG."""
    width, height, left, top = 760, 300, 70, 20
    longest, tallest = max(len(m) for m in MOVIES.values()), 400
    total, counted = sum(len(m) for m in MOVIES.values()), sum(len(v) for v in counts)
    x = lambda i: left + i * (width - left - 20) / (longest - 1)
    y = lambda v: top + (height - top - 50) * (1 - v / tallest)
    parts = [f'<svg viewBox="0 0 {width} {height + 50}" style="width: 100%; max-width: 900px; font: 16px sans-serif">',
             f'<line x1="{left}" y1="{y(0)}" x2="{width - 20}" y2="{y(0)}" stroke="#888"/>',
             f'<line x1="{left}" y1="{y(0)}" x2="{left}" y2="{top}" stroke="#888"/>']
    for v in (0, 100, 200, 300, 400):
        parts.append(f'<text x="{left - 8}" y="{y(v) + 5}" text-anchor="end" fill="#666">{v}</text>')
    parts.append(f'<text x="{left}" y="{height - 12}" fill="#666">frame</text>')
    parts.append(f'<text x="{left + 8}" y="{top + 14}" fill="#666">cells</text>')
    for values, colour, name in zip(counts, EMBRYO_COLOURS, MOVIES):
        points = " ".join(f"{x(i):.1f},{y(v):.1f}" for i, v in enumerate(values))
        parts.append(f'<polyline points="{points}" fill="none" stroke="{colour}" stroke-width="3.5"/>')
        if values:
            parts.append(f'<text x="{x(len(values) - 1) + 6:.1f}" y="{y(values[-1]) + 5:.1f}" fill="{colour}">{name}</text>')
    bar = (width - left - 20) * counted / total
    parts.append(f'<rect x="{left}" y="{height + 8}" width="{width - left - 20}" height="14" fill="#e8eaed"/>')
    parts.append(f'<rect x="{left}" y="{height + 8}" width="{bar:.1f}" height="14" fill="{COLORS[build]}"/>')
    weight = "bold" if done else "normal"
    parts.append(f'<text x="{width - 20}" y="{top + 30}" text-anchor="end" font-size="34" font-weight="{weight}" '
                 f'fill="{COLORS[build]}">{seconds:.1f} s</text>')
    parts.append(f'<text x="{width - 20}" y="{top + 54}" text-anchor="end" fill="#666">'
                 f'{build}: np.unique on {counted} of {total} frames</text>')
    return "".join(parts) + "</svg>"


def count_cells(build):
    """Count the cells of every frame in one build, drawing the result as it comes in."""
    blocks = json.dumps([(SHARED[name].name, MOVIES[name].dtype.str, MOVIES[name].shape) for name in MOVIES])
    worker = subprocess.Popen([BUILDS[build], "-c", MOVIE_WORKER, blocks], stdout=subprocess.PIPE, text=True)
    assert worker.stdout.readline().strip() == "ready"
    counts, seconds, drawn = [[] for _ in MOVIES], 0.0, 0.0
    shown = display(HTML(draw(build, counts, seconds, False)), display_id=True)
    for line in worker.stdout:
        index, cells, seconds = line.split()
        counts[int(index)].append(int(cells))
        seconds = float(seconds)
        if time.perf_counter() - drawn > 0.04:
            shown.update(HTML(draw(build, counts, seconds, False)))
            drawn = time.perf_counter()
    worker.wait()
    shown.update(HTML(draw(build, counts, seconds, True)))
    return {"seconds": seconds, "counts": counts}


RESULTS = {}

### On `main`

In [ ]:
RESULTS["main"] = count_cells("main")

### With the improvements

In [ ]:
RESULTS["tables"] = count_cells("tables")

In [ ]:
assert RESULTS["main"]["counts"] == RESULTS["tables"]["counts"]
fig, ax = plt.subplots(figsize=(8, 4))
for name, counts, colour in zip(MOVIES, RESULTS["tables"]["counts"], EMBRYO_COLOURS):
    ax.plot(np.arange(len(counts)) * FRAME_STEP, counts, color=colour, linewidth=3, label=name)
ax.set_xlabel("frame")
ax.set_ylabel("cells")
ax.legend(frameon=False)
ax.set_title("cells in each frame", loc="left")
plt.show()
bars({name: r["seconds"] for name, r in RESULTS.items()}, "s", "np.unique over both movies")
print("faster than main:", speedups(RESULTS))

## Why it is so much faster

In [ ]:
def unique_with_table(a):
    lo, hi = int(a.min()), int(a.max())     # one pass, and it can stop early
    if hi - lo >= a.size:                   # values too spread out: hash instead
        return None
    seen = np.zeros(hi - lo + 1, dtype=bool)
    seen[a - lo] = True                     # one write per element, nothing hashed
    return np.flatnonzero(seen) + lo        # read back in order: already sorted


small = np.array([7, 3, 7, 5, 3, 7])
print("values  ", small)
print("table   ", (np.bincount(small - small.min()) > 0).astype(int), " <- slots for", small.min(), "to", small.max())
print("unique  ", unique_with_table(small))

frame = MOVIES["embryo 1"][100 // FRAME_STEP].ravel()
assert np.array_equal(unique_with_table(frame), np.unique(frame))
print(f"\nmatches np.unique on a frame of the embryo movie ({frame.size:,} voxels)")
print("too spread out:", unique_with_table(np.array([1, 10**12, 5])))

Six lines of Python on `main` against `np.unique`.

In [ ]:
import inspect

PYTHON = "six lines of Python\non main"
rng = np.random.default_rng(0)
races = {"10 million int64, 1,000 distinct values": rng.integers(0, 1000, 10_000_000),
         "10 million int64, all distinct (shuffled 0 … n-1)": rng.permutation(10_000_000)}
for title, values in races.items():
    race = bench(values)
    python = bench(values, "unique_with_table(a)", define=inspect.getsource(unique_with_table), builds=["main"])
    ms = {"main": race["main"]["seconds"] * 1e3, PYTHON: python["main"]["seconds"] * 1e3,
          "tables": race["tables"]["seconds"] * 1e3}
    bars(ms, "ms", title)
    print(f"six lines of Python beat main's C++ by {ms['main'] / ms[PYTHON]:.0f}x; "
          f"the same idea in C++ is {ms[PYTHON] / ms['tables']:.1f}x faster again\n")

## More label data

### Relabel the neurons in this block

Fly brain, electron microscopy.

In [ ]:
CREMI = DATA / "cremi_neuron_ids.npy"
neurons = np.load(CREMI, mmap_mode="r")
print(f"{neurons.shape}  {neurons.dtype}  {neurons.nbytes / 2**20:,.0f} MB")

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
axes[0].imshow(np.load(DATA / "cremi_raw_slice.npy"), cmap="gray")
axes[0].set_title("electron microscopy", loc="left")
_, codes = np.unique(neurons[62], return_inverse=True)
axes[1].imshow(codes.reshape(neurons[62].shape) * 7919 % 20, cmap="tab20", interpolation="nearest")
axes[1].set_title("neuron labels", loc="left")
for ax in axes:
    ax.axis("off")
plt.show()

In [ ]:
seg = bench(CREMI, "np.unique(a, return_inverse=True)")
print(f"{seg['tables']['distinct']:,} neurons in {neurons.size:,} voxels\n")
bars({n: r["seconds"] for n, r in seg.items()}, "s", "relabel neurons to consecutive IDs (return_inverse): time")
bars({n: r["extra_bytes"] / 2**20 for n, r in seg.items()}, "MB", "relabel neurons: extra memory while it runs")
print("faster than main:", speedups(seg))

### How large is each brain region?

Mouse brain atlas. The IDs are too spread out for a table.

In [ ]:
ALLEN = DATA / "allen_annotation_25.npy"
atlas = np.load(ALLEN, mmap_mode="r")
show_labels(atlas[:, :, atlas.shape[2] // 2 - 40].T, f"{atlas.shape}  {atlas.dtype}  one coronal slice")

brain = bench(ALLEN, "np.unique(a, return_counts=True)")
print(f"{brain['tables']['distinct']:,} structures in {atlas.size:,} voxels\n")
bars({n: r["seconds"] for n, r in brain.items()}, "s", "volume of every brain region (return_counts): time")
print("faster than main:", speedups(brain))

### How much of each kind of land cover is in this tile?

One tile of 2,651.

In [ ]:
WORLDCOVER = DATA / "worldcover_N39W108.npy"
tile = np.load(WORLDCOVER, mmap_mode="r")
classes = {10: ("tree cover", "#006400"), 20: ("shrubland", "#ffbb22"), 30: ("grassland", "#ffff4c"),
           40: ("cropland", "#f096ff"), 50: ("built-up", "#fa0000"), 60: ("bare", "#b4b4b4"),
           70: ("snow and ice", "#f0f0f0"), 80: ("water", "#0064c8"), 90: ("wetland", "#0096a0"),
           95: ("mangroves", "#00cf75"), 100: ("moss and lichen", "#fae6a0")}
lut = np.zeros((256, 3))
for code, (_, colour) in classes.items():
    lut[code] = [int(colour[i:i + 2], 16) / 255 for i in (1, 3, 5)]
plt.figure(figsize=(6.5, 6.5))
plt.imshow(lut[np.asarray(tile[::40, ::40])], interpolation="nearest")
plt.axis("off")
plt.title(f"{tile.shape}  {tile.dtype}", loc="left")
plt.show()

land = bench(WORLDCOVER, "np.unique(a, return_counts=True)", repeats=2)
found = np.unique(np.asarray(tile[::40, ::40]))
print("classes present:", ", ".join(classes[v][0] for v in found if v in classes), "\n")
bars({n: r["seconds"] for n, r in land.items()}, "s", "area of every land cover class (return_counts): time")
print("faster than main:", speedups(land))
hours = {n: r["seconds"] * 2651 / 60 for n, r in land.items()}
print("the whole planet, 2,651 tiles: " + ", ".join(f"{n} {m:.0f} min" for n, m in hours.items()))

## Not only integers

In [ ]:
n = 10_000_000
rng = np.random.default_rng(0)
cases = {
    "datetime64[D]\n3,650 days": (19000 + rng.integers(0, 3650, n)).astype("M8[D]"),
    "S1\n16 categories": np.array(list("ACDEFGHIKLMNPQRS"), dtype="S1")[rng.integers(0, 16, n)],
    "int16\n1,000 codes": rng.integers(0, 1000, n).astype(np.int16),
    "complex128\n1,000 values": (rng.normal(size=1000) + 1j * rng.normal(size=1000))[rng.integers(0, 1000, n)],
}
other = {name: bench(a) for name, a in cases.items()}

fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(other))
width = 0.8 / len(BUILDS)
for i, build in enumerate(BUILDS):
    offset = (i - (len(BUILDS) - 1) / 2) * width
    ms = [r[build]["seconds"] * 1e3 for r in other.values()]
    ax.bar(x + offset, ms, width, label=build, color=COLORS[build])
    if build != "main":
        for j, r in enumerate(other.values()):
            ax.text(j + offset, ms[j], f"{r['main']['seconds'] / r[build]['seconds']:.0f}x",
                    ha="center", va="bottom", fontsize=12)
ax.set_xticks(x, list(other))
ax.set_ylabel("ms")
ax.legend(frameon=False)
ax.set_title("np.unique on 10 million elements", loc="left")
plt.show()


## The case that did not finish

`np.unique_values` on complex NaNs.

In [ ]:
sizes = [1_000, 2_000, 4_000, 8_000, 16_000, 32_000]
nan_runs = {n: bench(np.full(n, complex(np.nan, 0)), "np.unique_values(a)") for n in sizes}

fig, ax = plt.subplots(figsize=(8, 4))
for build in BUILDS:
    ax.plot(sizes, [nan_runs[n][build]["seconds"] * 1e3 for n in sizes], "o-", label=build,
            color=COLORS[build], linewidth=3)
ax.set_xlabel("number of NaNs")
ax.set_ylabel("ms")
ax.legend(frameon=False)
ax.set_title("np.unique_values on complex NaNs", loc="left")
plt.show()

last = nan_runs[sizes[-1]]
print(f"{sizes[-1]:,} NaNs: {last['main']['seconds'] * 1e3:,.0f} ms on main, "
      f"{last['tables']['seconds'] * 1e3:.2f} ms with the new hash set "
      f"({last['main']['seconds'] / last['tables']['seconds']:,.0f}x)")

## The whole workflow

The four analyses above, each read from the file its publisher distributes, then one `np.unique` call.

In [ ]:
# the movies are no longer needed: free their memory before the files are read again
for name in ("frame", "labels"):
    globals().pop(name, None)
MOVIES.clear()
for block in SHARED.values():
    block.close()
    block.unlink()
SHARED.clear()

import summary_plot

saved = summary_plot.run()      # a few minutes; run(replot=True) redraws the timings of the last run
for key, workflow in summary_plot.WORKFLOWS.items():
    main, new = saved["timings"][key]["main"], saved["timings"][key]["this PR"]
    print(f"{workflow['title']:30s} {workflow['call']:36s} {main['unique'] / new['unique']:5.1f}x faster, "
          f"{sum(main.values()) / sum(new.values()):4.1f}x with I/O")
figure = summary_plot.draw(saved)
figure.savefig("summary.png", dpi=300, facecolor="white")

## Data

- CREMI sample A: J. Funke, S. Saalfeld, D. Bock, S. Turaga, E. Perlman, *MICCAI Challenge on Circuit Reconstruction from Electron Microscopy Images* (2016), https://cremi.org
- Allen Mouse Brain CCFv3: Q. Wang et al., *The Allen Mouse Brain Common Coordinate Framework: A 3D Reference Atlas*, Cell 181 (2020). © Allen Institute for Brain Science, used under its terms for noncommercial use.

- Fluo-N3DH-CE: *C. elegans* embryo movies from the Waterston Lab, University of Washington, with the silver-truth segmentations of the Cell Tracking Challenge: M. Maška et al., *The Cell Tracking Challenge: 10 years of objective benchmarking*, Nature Methods 20 (2023), https://celltrackingchallenge.net
- ESA WorldCover 10 m 2021 v200: Zanaga et al. (2022), https://doi.org/10.5281/zenodo.7254221. © ESA WorldCover project, contains modified Copernicus Sentinel data (2021), CC BY 4.0.

All four are downloaded from their publishers by `pixi run data`. The arrays in the other sections are generated.